# Assignment Sesi 29 Tugas 2
Nama: Faraday Barr Fatahillah

**Tugas 2**

Anda bekerja di perusahaan Otomotif dan diberikan link [ini](https://autocatalogarchive.com/mitsubishi/) yang berisi informasi tentang beberapa mobil mitsubishi dengan berbagai macam bahasa dalam bentuk dokumen PDF. Dari katolog tersebut, gunakanlah semua file PDF yang berbahasa Indonesia atau berkode (ID), misalkan **2018 - Outlander Sport (ID)**.

Buatlah AI yang dapat yang dapat melakukan Product Search yang dapat menjawab atau mencari konteks yang cocok dengan input berikut:
- Detail spesifikasi Mitsubishi Destinator
- Mobil yang cocok untuk Travel dengan jumlah bangku atau *seating capacity* yang besar.
- Mobil untuk perjalanan jauh yang nyaman
- Mobil Mitsubishi yang irit bahan bakar
- Mobil Mitsubishi hybrid atau electric 

Lakukanlah pencarian dengan Hybrid Search dan gunakanlah Pinecone sebagai vector database.

In [ ]:
import re
import os
import json
import pymupdf
import chromadb
import pdfplumber
import numpy as np
from tqdm import tqdm
from pathlib import Path
from rank_bm25 import BM25Okapi
from chromadb.config import Settings
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer


from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak, HRFlowable
)
from reportlab.lib import colors

In [ ]:
PDF_FOLDER = "/Session29_Tugas2_PDFS"

COMBINED_PDF_PATH = "/tmp/mitsubishi_combined.pdf"
CHROMA_PERSIST_DIR = "/tmp/chroma_mitsubishi"
CHROMA_COLLECTION = "mitsubishi_id"

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "YOUR_PINECONE_API_KEY")
PINECONE_ENV = os.getenv("PINECONE_ENV", "us-east-1")
PINECONE_INDEX = "mitsubishi-id"

EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
EMBED_DIM = 384

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

ALPHA = 0.6
TOP_K = 5

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    pages_text = []
    for page in doc:
        text = page.get_text('text').strip()
        if text:
            pages_text.append(text)
    doc.close()

    tables_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    clean = [cell.strip() if cell else '' for cell in row]
                    if any(clean):
                        tables_text.append(' | '.join(clean))

    combined = '\n'.join(pages_text)
    if tables_text:
        combined += '\n\n--- TABLES ---\n' + '\n'.join(tables_text)

    return re.sub(r'\n{3,}', '\n\n', combined).strip()


pdf_files = sorted(Path(PDF_FOLDER).glob('*.pdf'))
print(f'Found {len(pdf_files)} PDF(s):')
for p in pdf_files:
    print(f'  • {p.name}')

In [ ]:
brochures: list[dict] = []

for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    title = pdf_path.stem
    text = extract_text_from_pdf(str(pdf_path))
    brochures.append({"title": title, "path": str(pdf_path), "text": text})
    print(f"[{title}] → {len(text):,} chars")

print(f"\nTotal brochures parsed: {len(brochures)}")

In [ ]:
def build_combined_pdf(brochures: list[dict], output_path: str) -> None:
    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        rightMargin=2*cm, leftMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "BrochureTitle",
        parent=styles["Heading1"],
        fontSize=18,
        textColor=colors.HexColor("#C8102E"),
        spaceAfter=12,
    )
    body_style = ParagraphStyle(
        "BrochureBody",
        parent=styles["Normal"],
        fontSize=9,
        leading=13,
        spaceAfter=4,
    )

    story = []
    for i, brochure in enumerate(brochures):
        if i > 0:
            story.append(PageBreak())

        story.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor("#C8102E")))
        story.append(Spacer(1, 6))
        story.append(Paragraph(f"BROCHURE: {brochure['title']}", title_style))
        story.append(HRFlowable(width="100%", thickness=1, color=colors.grey))
        story.append(Spacer(1, 12))

        safe_text = (
            brochure["text"]
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
        )
        for para in safe_text.split("\n\n"):
            para = para.strip()
            if para:
                story.append(Paragraph(para.replace("\n", "<br/>"), body_style))
                story.append(Spacer(1, 4))

    doc.build(story)
    print(f"Combined PDF saved → {output_path}")

build_combined_pdf(brochures, COMBINED_PDF_PATH)

In [ ]:
combined_text = extract_text_from_pdf(COMBINED_PDF_PATH)
print(f"Combined PDF total characters: {len(combined_text):,}")
print("\n--- Preview (first 500 chars) ---")
print(combined_text[:500])

In [ ]:
def recursive_split(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    separators: list[str] | None = None,
) -> list[str]:
    if separators is None:
        separators = ["\n\n", "\n"]

    def _split(text: str, seps: list[str]) -> list[str]:
        if len(text) <= chunk_size:
            return [text] if text.strip() else []

        sep = seps[0] if seps else ""
        parts = text.split(sep) if sep else list(text)

        chunks: list[str] = []
        current = ""
        for part in parts:
            candidate = (current + sep + part) if current else part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current.strip():
                    if len(current) > chunk_size and len(seps) > 1:
                        chunks.extend(_split(current, seps[1:]))
                    else:
                        chunks.append(current)
                current = part
        if current.strip():
            if len(current) > chunk_size and len(seps) > 1:
                chunks.extend(_split(current, seps[1:]))
            else:
                chunks.append(current)
        return chunks

    raw_chunks = _split(text, separators)

    if chunk_overlap == 0 or len(raw_chunks) <= 1:
        return raw_chunks

    overlapped: list[str] = [raw_chunks[0]]
    for i in range(1, len(raw_chunks)):
        tail = overlapped[-1][-chunk_overlap:]
        merged = (tail + " " + raw_chunks[i]).strip()
        overlapped.append(merged[:chunk_size + chunk_overlap])
    return overlapped

all_chunks = recursive_split(combined_text)
print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk length: {sum(len(c) for c in all_chunks)/len(all_chunks):.0f} chars")
print("\n--- Sample chunk ---")
print(all_chunks[0])

In [ ]:
def tag_chunks_with_source(
    chunks: list[str],
    brochures: list[dict],
) -> list[dict]:
    current_source = "unknown"
    tagged: list[dict] = []

    for i, chunk in enumerate(chunks):
        for brochure in brochures:
            if f"BROCHURE: {brochure['title']}" in chunk:
                current_source = brochure["title"]
                break
        tagged.append({
            "id":     f"chunk_{i:05d}",
            "text":   chunk.strip(),
            "source": current_source,
        })
    return tagged


tagged_chunks = tag_chunks_with_source(all_chunks, brochures)
print(f"Tagged {len(tagged_chunks)} chunks.")

from collections import Counter
dist = Counter(c["source"] for c in tagged_chunks)
for src, count in dist.most_common():
    print(f"  {src}: {count} chunks")

In [ ]:
print(f"Loading embedding model: {EMBED_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

texts = [c["text"] for c in tagged_chunks]

print(f"Embedding {len(texts)} chunks (this may take a minute) ...")
embeddings = embed_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
chroma_client = chromadb.PersistentClient(
    path=CHROMA_PERSIST_DIR,
    settings=Settings(anonymized_telemetry=False),
)

try:
    chroma_client.delete_collection(CHROMA_COLLECTION)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

BATCH = 100
for start in tqdm(range(0, len(tagged_chunks), BATCH), desc="ChromaDB upsert"):
    batch = tagged_chunks[start : start + BATCH]
    collection.add(
        ids = [c["id"]   for c in batch],
        documents = [c["text"] for c in batch],
        embeddings = embeddings[start : start + BATCH].tolist(),
        metadatas = [{"source": c["source"]} for c in batch],
    )

print(f"ChromaDB collection '{CHROMA_COLLECTION}' → {collection.count()} documents")

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [idx.name for idx in pc.list_indexes()]
if PINECONE_INDEX not in existing_indexes:
    pc.create_index(
        name = PINECONE_INDEX,
        dimension = EMBED_DIM,
        metric = "cosine",
        spec = ServerlessSpec(cloud="aws", region=PINECONE_ENV),
    )
    print(f"Created Pinecone index '{PINECONE_INDEX}'")
else:
    print(f"Using existing Pinecone index '{PINECONE_INDEX}'")

index = pc.Index(PINECONE_INDEX)

BATCH = 100
for start in tqdm(range(0, len(tagged_chunks), BATCH), desc="Pinecone upsert"):
    batch = tagged_chunks[start : start + BATCH]
    embeds = embeddings[start : start + BATCH]
    vectors = [
        {
            "id": chunk["id"],
            "values": emb.tolist(),
            "metadata": {"text": chunk["text"], "source": chunk["source"]},
        }
        for chunk, emb in zip(batch, embeds)
    ]
    index.upsert(vectors=vectors)

stats = index.describe_index_stats()
print(f"Pinecone index stats: {stats}")

In [ ]:
def tokenize(text: str) -> list[str]:
    return re.sub(r"[^\w\s]", " ", text.lower()).split()

corpus_tokens = [tokenize(c["text"]) for c in tagged_chunks]
bm25 = BM25Okapi(corpus_tokens)
print(f"BM25 index built over {len(corpus_tokens)} documents.")

In [ ]:
def min_max_norm(arr: np.ndarray) -> np.ndarray:
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-9)


def hybrid_search(
    query: str,
    top_k: int = TOP_K,
    alpha: float = ALPHA,
    use_pinecone: bool = False,
) -> list[dict]:
    n = len(tagged_chunks)

    query_emb = embed_model.encode(
        [query], normalize_embeddings=True
    )[0]

    if use_pinecone:
        pass

    dense_scores_raw = embeddings @ query_emb
    dense_scores = min_max_norm(dense_scores_raw)

    query_tokens    = tokenize(query)
    bm25_scores_raw = np.array(bm25.get_scores(query_tokens))
    bm25_scores     = min_max_norm(bm25_scores_raw)

    hybrid_scores = alpha * dense_scores + (1 - alpha) * bm25_scores

    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        chunk = tagged_chunks[idx]
        results.append({
            "rank": rank,
            "id": chunk["id"],
            "source": chunk["source"],
            "hybrid_score": float(hybrid_scores[idx]),
            "dense_score": float(dense_scores_raw[idx]),
            "bm25_score": float(bm25_scores_raw[idx]),
            "text": chunk["text"],
        })
    return results


def pretty_print_results(query: str, results: list[dict]) -> None:
    sep = "=" * 80
    print(f"\n{sep}")
    print(f"QUERY: {query}")
    print(sep)
    for r in results:
        print(f"\n[Rank {r['rank']}] Source: {r['source']}")
        print(f"Hybrid={r['hybrid_score']:.4f}  Dense={r['dense_score']:.4f}  BM25={r['bm25_score']:.4f}")
        print(f"ID: {r['id']}")
        print(f"Text preview: {r['text'][:300].replace(chr(10), ' ')} ...")
    print()


print("Hybrid search function ready.")

In [ ]:
Q1 = "Detail spesifikasi Mitsubishi Destinator"
results_q1 = hybrid_search(Q1, top_k=TOP_K)
pretty_print_results(Q1, results_q1)

In [ ]:
Q2 = "Mobil yang cocok untuk Travel dengan jumlah bangku atau seating capacity besar"
results_q2 = hybrid_search(Q2, top_k=TOP_K)
pretty_print_results(Q2, results_q2)

In [ ]:
Q3 = "Mobil untuk perjalanan jauh yang nyaman"
results_q3 = hybrid_search(Q3, top_k=TOP_K)
pretty_print_results(Q3, results_q3)

In [ ]:
Q4 = "Mobil Mitsubishi yang irit bahan bakar"
results_q4 = hybrid_search(Q4, top_k=TOP_K)
pretty_print_results(Q4, results_q4)

In [ ]:
Q5 = "Mobil Mitsubishi hybrid atau electric"
results_q5 = hybrid_search(Q5, top_k=TOP_K)
pretty_print_results(Q5, results_q5)

In [ ]:
summary = {
    "Q1 - Spesifikasi Destinator":  [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q1],
    "Q2 - Seating Capacity Besar": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q2],
    "Q3 - Perjalanan Jauh Nyaman": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q3],
    "Q4 - Irit Bahan Bakar": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q4],
    "Q5 - Hybrid atau Electric": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q5]
}

print(json.dumps(summary, indent=2, ensure_ascii=False))